# Snowpark Notebook: Load Files into STAGING

This notebook loads CSV/Parquet files from Snowflake stages into `STAGING` tables.

In [ ]:
import os
import re
import pandas as pd
from typing import Optional
from snowflake.snowpark import Session


def _env(name: str, default: Optional[str] = None) -> str:
    value = os.getenv(name, default)
    if value is None:
        raise EnvironmentError(f"Missing required environment variable: {name}")
    return value


connection_parameters = {
    'account': _env('SNOWFLAKE_ACCOUNT'),
    'user': _env('SNOWFLAKE_USER'),
    'password': _env('SNOWFLAKE_PASSWORD'),
    'role': _env('SNOWFLAKE_ROLE', 'ACCOUNTADMIN'),
    'warehouse': _env('SNOWFLAKE_WAREHOUSE', 'ADVENTUREWORKS_ETL_WH'),
    'database': _env('SNOWFLAKE_DATABASE', 'ADVENTUREWORKS_MIGRATED'),
}

DATABASE = connection_parameters['database']
STAGING_SCHEMA = 'STAGING'
PARQUET_STAGE = '@ADVENTUREWORKS_MIGRATED.STAGING.parquet_internal_stage'
CSV_STAGE = '@ADVENTUREWORKS_MIGRATED.STAGING.csv_internal_stage'
CSV_FORMAT = 'ADVENTUREWORKS_MIGRATED.UTILITY.ff_csv'
FULL_REFRESH = False

session = Session.builder.configs(connection_parameters).create()
session.sql('SELECT CURRENT_WAREHOUSE() AS warehouse').show()


In [ ]:
def quote_ident(identifier: str) -> str:
    return '"' + identifier.replace('"', '""') + '"'


def fqtn(schema: str, table: str) -> str:
    return f"{quote_ident(DATABASE)}.{quote_ident(schema)}.{quote_ident(table)}"


def table_base_name(staging_table: str) -> str:
    if staging_table.upper().endswith('_STG'):
        return staging_table[:-4]
    return staging_table


def pattern_for(base_name: str, extension: str) -> str:
    escaped = re.escape(base_name)
    return rf"(?i).*{escaped}.*\.{extension}"


def has_files(stage: str, pattern: str) -> bool:
    rows = session.sql(f"LIST {stage} PATTERN = '{pattern}'").collect()
    return len(rows) > 0


staging_tables = [
    row['TABLE_NAME']
    for row in session.sql(
        f"""
        SELECT TABLE_NAME
        FROM {quote_ident(DATABASE)}.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = '{STAGING_SCHEMA}'
          AND TABLE_TYPE = 'BASE TABLE'
        ORDER BY TABLE_NAME
        """
    ).collect()
]

pd.DataFrame({'staging_tables': staging_tables})


In [ ]:
load_results = []

for table_name in staging_tables:
    target_fqn = fqtn(STAGING_SCHEMA, table_name)
    base_name = table_base_name(table_name)

    parquet_pattern = pattern_for(base_name, 'parquet')
    csv_pattern = pattern_for(base_name, 'csv')

    parquet_available = has_files(PARQUET_STAGE, parquet_pattern)
    csv_available = has_files(CSV_STAGE, csv_pattern)

    if FULL_REFRESH:
        session.sql(f"TRUNCATE TABLE {target_fqn}").collect()

    load_mode = 'SKIPPED'
    copy_result = {}

    if parquet_available:
        copy_sql = f"""
          COPY INTO {target_fqn}
          FROM {PARQUET_STAGE}
          PATTERN = '{parquet_pattern}'
          MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
          ON_ERROR = 'CONTINUE'
        """
        copy_rows = session.sql(copy_sql).collect()
        copy_result = copy_rows[0].as_dict() if copy_rows else {}
        load_mode = 'PARQUET'
    elif csv_available:
        copy_sql = f"""
          COPY INTO {target_fqn}
          FROM {CSV_STAGE}
          PATTERN = '{csv_pattern}'
          FILE_FORMAT = (FORMAT_NAME = '{CSV_FORMAT}')
          ON_ERROR = 'CONTINUE'
        """
        copy_rows = session.sql(copy_sql).collect()
        copy_result = copy_rows[0].as_dict() if copy_rows else {}
        load_mode = 'CSV'

    row_count = session.sql(f"SELECT COUNT(*) AS CNT FROM {target_fqn}").collect()[0]['CNT']

    load_results.append(
        {
            'table_name': table_name,
            'base_name': base_name,
            'parquet_available': parquet_available,
            'csv_available': csv_available,
            'load_mode': load_mode,
            'row_count_after_load': row_count,
            'copy_result': copy_result,
        }
    )

load_df = pd.DataFrame(load_results)
load_df


In [ ]:
session.close()
print('Staging load notebook execution completed.')
